# 📈 Bayesian Optimization for S&P 500 Pair Trading — Walk-Forward Edition
### Optuna TPE · Cointegration · Z-Score Mean Reversion · Quarterly Walk-Forward Re-Optimization

**Author:** Derren Winata | NUS Data Science & Analytics  
**Stack:** Python · Optuna · statsmodels · yfinance · Plotly  

---

## 🗺️ Overview

This project applies **Bayesian Optimization (BO)** via Optuna to tune a cointegration-based pair trading strategy — with **quarterly walk-forward re-optimization** so parameters adapt to changing market regimes rather than being frozen from one historical fit.

### Market Regimes Covered (2016 – 2025)
| Period | Regime |
|--------|--------|
| 2016–2019 | Pre-COVID bull run |
| 2020 | COVID crash & V-shaped recovery |
| 2021–2022 | Rate-hike cycle & tech drawdown |
| 2023–2025 | AI-driven rally & high-rate environment |

### Upgrades Over Naive BO Backtests
| Feature | Naive Approach | This Project |
|---------|---------------|--------------|
| Parameter fitting | Once on full history | **Quarterly walk-forward re-optimization** |
| Universe | 30 tickers | **42 tickers, 6 sectors** |
| Transaction costs | Not modelled | **2 bps per trade leg** |
| Cointegration | Engle-Granger only | **Engle-Granger + Johansen** |
| Risk management | None | **Stop-loss z-score parameter** |
| Parameters optimized | 3 | **4** |

### Parameters Optimized
| Parameter | Range | Description |
|-----------|-------|-------------|
| `lookback` | 20–120 days | Rolling window for hedge ratio & z-score |
| `entry_z` | 1.0–3.5 σ | Enter when spread deviates this far |
| `exit_z` | 0.0–1.0 σ | Exit when spread reverts this close to mean |
| `stop_z` | 3.0–6.0 σ | Stop-loss: exit if spread blows out this far |

---

## 📦 Setup — Libraries & Configuration

In [1]:
import warnings
warnings.filterwarnings('ignore')

import yfinance as yf
import pandas as pd
import numpy as np

from statsmodels.tsa.stattools import coint
from statsmodels.tsa.vector_ar.vecm import coint_johansen
import statsmodels.api as sm

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from itertools import combinations
from IPython.display import display
from dateutil.relativedelta import relativedelta
from datetime import datetime
from joblib import Parallel, delayed

print("✅  All libraries loaded")
print(f"   optuna     v{optuna.__version__}")
print(f"   pandas     v{pd.__version__}")

✅  All libraries loaded
   optuna     v4.2.1
   pandas     v2.2.2


In [2]:
# ================================================================
# CONFIGURATION
# ================================================================

# Expanded universe: 42 tickers across 6 sectors (7 per sector)
# — classic within-sector competitor pairs added for stronger cointegration signal
TICKERS = {
    'Energy':     ['XOM', 'CVX', 'COP', 'PSX', 'SLB', 'EOG', 'OXY'],
    'Financials': ['JPM', 'BAC', 'GS',  'MS',  'WFC', 'C',   'BLK'],
    'Consumer':   ['KO',  'PEP', 'MCD', 'YUM', 'SBUX','NKE', 'CL'],
    'Tech':       ['MSFT','GOOGL','META','AMZN','ORCL','NVDA','INTC'],
    'Healthcare': ['JNJ', 'ABT', 'MDT', 'BMY', 'PFE', 'MRK', 'LLY'],
    'Industrial': ['HON', 'MMM', 'GE',  'CAT', 'UPS', 'BA',  'RTX'],
}

# Full date range
FULL_START  = '2016-01-01'
FULL_END    = '2025-12-31'

# Walk-forward splits
WARMUP_END  = '2018-12-31'   # First BO training window ends here
WF_STEP     = 3              # Months between re-optimization (quarterly)
WF_TRAIN_YRS = 2             # Years of history used for each BO fit

# Cointegration — relaxed to 0.10 to include economically-motivated within-sector pairs
COINT_PVALUE = 0.10

# Cointegration — sector-only constraint (within-sector pairs only)
SECTOR_ONLY_PAIRS = True   # Restrict to economically-justified same-sector pairs

# Bayesian Optimization
N_TRIALS         = 150
N_CV_FOLDS       = 5
N_ADAPTIVE_TRIALS = 50   # Reduced trials for adaptive portfolio re-selection runs

# Parallel execution: run pairs simultaneously using joblib
USE_PARALLEL = True

# Transaction costs (bps per leg; 1 full round-trip = 4 bps total)
TRANSACTION_COST_BPS = 2.0

# Baseline parameters (fixed, never re-optimized)
BASELINE_PARAMS = {'lookback': 60, 'entry_z': 2.0, 'exit_z': 0.5, 'stop_z': 4.5}

# Top N pairs for individual analysis
TOP_N_PAIRS = 5
# Top N pairs per year for adaptive portfolio
TOP_N_ADAPTIVE = 2

import os
OUTPUT_DIR = "outputs"
os.makedirs(f"{OUTPUT_DIR}/charts", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/data",   exist_ok=True)

print("✅  Configuration set")
print(f"   Universe      : {sum(len(v) for v in TICKERS.values())} tickers across {len(TICKERS)} sectors")
print(f"   Full range    : {FULL_START} → {FULL_END}")
print(f"   WF step       : every {WF_STEP} months (quarterly re-optimization)")
print(f"   Trans. cost   : {TRANSACTION_COST_BPS} bps per leg")
print(f"   Sector-only   : {SECTOR_ONLY_PAIRS}")
print(f"   Parallel BO   : {USE_PARALLEL}")
print(f"   Output dir    : ./{OUTPUT_DIR}/")

✅  Configuration set
   Universe      : 42 tickers across 6 sectors
   Full range    : 2016-01-01 → 2025-12-31
   WF step       : every 3 months (quarterly re-optimization)
   Trans. cost   : 2.0 bps per leg
   Sector-only   : True
   Parallel BO   : True
   Output dir    : ./outputs/


---
## 📥 Data Download

In [3]:
all_tickers = [t for v in TICKERS.values() for t in v]
print(f"Downloading {len(all_tickers)} tickers: {FULL_START} → {FULL_END}...")

raw = yf.download(all_tickers, start=FULL_START, end=FULL_END,
                  auto_adjust=True, progress=False)

# Support both flat and MultiIndex column formats across yfinance versions
if isinstance(raw.columns, pd.MultiIndex):
    prices = raw['Close'].copy()
else:
    prices = raw[['Close']].copy() if 'Close' in raw.columns else raw.copy()

# Flatten single-ticker edge case
if isinstance(prices, pd.Series):
    prices = prices.to_frame()

prices = prices.dropna(axis=1, how='any')

# Flag dropped tickers
dropped = set(all_tickers) - set(prices.columns)
if dropped:
    print(f"⚠️  Dropped {len(dropped)} tickers with missing data: {dropped}")

print(f"\n✅  {prices.shape[1]} tickers, {prices.shape[0]} trading days")
print(f"   Date range: {prices.index[0].date()} → {prices.index[-1].date()}")
prices.tail(3)


✅  42 tickers, 2513 trading days
   Date range: 2016-01-04 → 2025-12-30


Ticker,ABT,AMZN,BA,BAC,BLK,BMY,C,CAT,CL,COP,...,PEP,PFE,PSX,RTX,SBUX,SLB,UPS,WFC,XOM,YUM
Date,,,,,,,,,,,,,,,,,,,,,
2025-12-26,124.210808,232.520004,216.440002,56.169998,1088.109985,54.001823,119.795578,581.639160,79.244751,90.833130,...,143.779999,24.676640,127.750000,184.557007,84.534843,37.568459,99.157196,94.794106,118.321342,152.536987
2025-12-29,123.942169,232.070007,217.250000,55.349998,1088.400024,53.478016,117.517448,577.259399,79.204994,91.914703,...,144.240005,24.588123,128.589996,183.809494,85.021698,37.677814,98.309021,94.057716,119.731941,151.571426
2025-12-30,125.146072,232.529999,218.500000,55.279999,1083.310059,53.576847,116.602219,576.042236,79.026093,93.373360,...,144.160004,24.578287,129.559998,183.400833,84.703751,38.324001,98.269577,93.848747,120.188896,151.481842


---
## 🔬 Step 1 — Dual Cointegration Testing

We use **both** Engle-Granger and Johansen tests. A pair must pass at least one to qualify.

**Engle-Granger:** Regress price_A on price_B → test residual stationarity (ADF).  
Order-dependent, but simple and interpretable.

**Johansen:** Models both series jointly via a VECM. Order-independent and more powerful
for detecting cointegration when both series have comparable unit roots.

A pair is accepted if **either** test has p-value < 0.05.

In [4]:
def eg_test(price_a: pd.Series, price_b: pd.Series) -> dict:
    """Engle-Granger: returns pvalue, hedge_ratio, half_life."""
    _, pvalue, _ = coint(price_a, price_b)

    X     = sm.add_constant(price_b)
    model = sm.OLS(price_a, X).fit()
    beta  = model.params.iloc[1]   # slope — positional to avoid name=None bug (pandas 2.x)

    spread     = (price_a - beta * price_b).rename('spread')
    sp_lag     = spread.shift(1).dropna()
    sp_diff    = spread.diff().dropna()
    m_hl       = sm.OLS(sp_diff, sm.add_constant(sp_lag)).fit()
    lam        = m_hl.params.iloc[1]   # positional — sp_lag.name is 'spread', but safe either way
    half_life  = round(-np.log(2) / lam, 1) if lam < 0 else None

    return {'eg_pvalue': round(pvalue, 4), 'hedge_ratio': round(beta, 4),
            'half_life': half_life}


def johansen_test(price_a: pd.Series, price_b: pd.Series) -> float:
    """Johansen trace test: returns approximate p-value (0.01, 0.05, or 0.10)."""
    try:
        data   = pd.concat([price_a, price_b], axis=1).dropna()
        result = coint_johansen(data, det_order=0, k_ar_diff=1)
        # Critical values at index 0=90%, 1=95%, 2=99%
        trace_stat = result.lr1[0]
        crit_95    = result.cvt[0, 1]
        crit_99    = result.cvt[0, 2]
        if   trace_stat > crit_99: return 0.01
        elif trace_stat > crit_95: return 0.05
        else:                      return 0.15
    except Exception:
        return 1.0


# Reverse-lookup: ticker → sector name (used for sector-only filtering)
SECTOR_MAP = {ticker: sector for sector, tickers in TICKERS.items() for ticker in tickers}


def find_cointegrated_pairs(prices: pd.DataFrame,
                             significance: float = COINT_PVALUE,
                             sector_only: bool   = SECTOR_ONLY_PAIRS) -> pd.DataFrame:
    """
    Test pairs with EG + Johansen; keep if either passes.
    sector_only=True restricts to within-sector pairs (more economically defensible).
    """
    tickers   = prices.columns.tolist()
    all_pairs = [(t1, t2) for t1, t2 in combinations(tickers, 2)
                 if not sector_only or SECTOR_MAP.get(t1) == SECTOR_MAP.get(t2)]
    total     = len(all_pairs)
    mode      = "within-sector" if sector_only else "all-cross"
    print(f"Testing {total} {mode} pairs (EG + Johansen, threshold p < {significance})...")
    results = []

    for t1, t2 in all_pairs:
        try:
            eg  = eg_test(prices[t1], prices[t2])
            j_p = johansen_test(prices[t1], prices[t2])
            passes_eg = eg['eg_pvalue'] < significance
            passes_jo = j_p          < significance

            if passes_eg or passes_jo:
                results.append({
                    'ticker_1':    t1,
                    'ticker_2':    t2,
                    'label':       f"{t1}/{t2}",
                    'eg_pvalue':   eg['eg_pvalue'],
                    'jo_pvalue':   j_p,
                    'passes_both': passes_eg and passes_jo,
                    'hedge_ratio': eg['hedge_ratio'],
                    'half_life':   eg['half_life'],
                })
        except Exception:
            pass

    df = pd.DataFrame(results)
    if df.empty:
        print("⚠️  No cointegrated pairs found — try a different period or lower threshold.")
        return df

    # Sort: pairs passing both tests first, then by EG p-value
    df = df.sort_values(['passes_both', 'eg_pvalue'], ascending=[False, True]).reset_index(drop=True)
    both = df['passes_both'].sum()
    print(f"\n✅  {len(df)} pairs pass at least one test  ({both} pass both EG + Johansen)")
    return df


# Run on full warmup period (2016–2018) for initial pair selection
warmup_prices = prices.loc[FULL_START:WARMUP_END]
coint_pairs   = find_cointegrated_pairs(warmup_prices)
display(coint_pairs.head(10))

Testing 126 within-sector pairs (EG + Johansen, threshold p < 0.1)...



✅  8 pairs pass at least one test  (1 pass both EG + Johansen)


,ticker_1,ticker_2,label,eg_pvalue,jo_pvalue,passes_both,hedge_ratio,half_life
0,META,NVDA,META/NVDA,0.0881,0.01,True,12.7836,27.0
1,GS,WFC,GS/WFC,0.0555,0.15,False,7.1000,31.8
2,GOOGL,NVDA,GOOGL/NVDA,0.0582,0.15,False,3.8284,24.6
3,MDT,PFE,MDT/PFE,0.0818,0.15,False,1.6215,22.8
4,PEP,YUM,PEP/YUM,0.0902,0.15,False,0.4555,37.7
5,RTX,UPS,RTX/UPS,0.1142,0.05,False,0.9217,39.3
6,COP,CVX,COP/CVX,0.4034,0.05,False,0.8240,63.3
7,BA,CAT,BA/CAT,0.5181,0.05,False,3.1846,60.8


---
## ⚙️ Step 2 — Strategy Engine (with Transaction Costs & Stop-Loss)

Two upgrades over the naive version:

**Transaction costs:** Each entry and exit involves 2 legs (buy A + sell B, or vice versa).  
Applied as `cost_bps × 2` on the spread return at each trade to realistically model friction.

**Stop-loss:** If the spread blows out beyond `stop_z` standard deviations, the position is  
closed immediately — protecting against de-cointegration events (mergers, spinoffs, sector breaks).

In [5]:
COST_FACTOR = TRANSACTION_COST_BPS / 10_000  # bps → decimal, per leg


class PairTradingStrategy:
    """
    Z-score mean reversion with transaction costs and stop-loss.

    Parameters
    ----------
    lookback : int    Rolling window for hedge ratio & z-score
    entry_z  : float  Enter when |z| > entry_z
    exit_z   : float  Exit  when |z| < exit_z (mean reversion)
    stop_z   : float  Stop-loss: exit if |z| > stop_z (de-cointegration guard)
    cost_bps : float  Transaction cost per leg in bps
    """

    def __init__(self, lookback=60, entry_z=2.0, exit_z=0.5, stop_z=4.5,
                 cost_bps=TRANSACTION_COST_BPS):
        self.lookback = int(lookback)
        self.entry_z  = float(entry_z)
        self.exit_z   = float(exit_z)
        self.stop_z   = float(stop_z)
        self.cost     = float(cost_bps) / 10_000

    def _rolling_hedge(self, pa: pd.Series, pb: pd.Series) -> pd.Series:
        hedge, lb = pd.Series(np.nan, index=pa.index), self.lookback
        for i in range(lb, len(pa)):
            y = pa.iloc[i-lb:i].values
            x = pb.iloc[i-lb:i].values
            X = np.column_stack([np.ones(lb), x])
            try:
                hedge.iloc[i] = np.linalg.lstsq(X, y, rcond=None)[0][1]
            except Exception:
                pass
        return hedge

    def _zscore(self, spread: pd.Series) -> pd.Series:
        m = spread.rolling(self.lookback).mean()
        s = spread.rolling(self.lookback).std().replace(0, np.nan)
        return (spread - m) / s

    def run(self, pa: pd.Series, pb: pd.Series) -> dict:
        hedge  = self._rolling_hedge(pa, pb)
        spread = pa - hedge * pb
        z      = self._zscore(spread)

        pos, pos_arr  = 0, np.zeros(len(z))
        trades        = []   # (index, direction) for cost accounting

        for i in range(self.lookback, len(z)):
            zi = z.iloc[i]
            if np.isnan(zi):
                continue
            prev_pos = pos
            if pos == 0:
                if   zi < -self.entry_z: pos =  1
                elif zi >  self.entry_z: pos = -1
            elif pos ==  1:
                if zi > -self.exit_z or zi > self.stop_z:  pos = 0
            elif pos == -1:
                if zi <  self.exit_z or zi < -self.stop_z: pos = 0
            pos_arr[i] = pos
            if pos != prev_pos:
                trades.append(i)

        positions = pd.Series(pos_arr, index=pa.index)

        # ── Dollar-neutral returns ──────────────────────────────────────────
        # Each day's P&L: long 1 unit of A, short hedge_ratio units of B.
        # Normalized by prior-day gross capital deployed (long + short legs)
        # so returns are directly comparable across pairs and meaningful in %.
        ret_a  = pa.pct_change().fillna(0)
        ret_b  = pb.pct_change().fillna(0)
        h_prev = hedge.shift(1)                         # use prior-day hedge ratio
        pnl    = ret_a * pa.shift(1) - h_prev * ret_b * pb.shift(1)
        gross  = (pa.shift(1) + h_prev.abs() * pb.shift(1)).replace(0, np.nan)
        dn_ret = (pnl / gross).fillna(0)
        strat_ret = positions.shift(1).fillna(0) * dn_ret

        # Deduct transaction costs at each trade (2 legs × cost_bps)
        cost_series = pd.Series(0.0, index=pa.index)
        for ti in trades:
            if ti < len(cost_series):
                cost_series.iloc[ti] = self.cost * 2
        strat_ret = strat_ret - cost_series

        return {'returns': strat_ret, 'positions': positions,
                'zscore': z, 'spread': spread, 'hedge': hedge,
                'n_trades': len(trades)}

    @staticmethod
    def metrics(returns: pd.Series) -> dict:
        rets = returns.dropna()
        if len(rets) < 10 or rets.std() == 0:
            return dict(sharpe=-99, sortino=-99, max_drawdown=-1.,
                        ann_return=-1., total_return=-1., calmar=0.,
                        pct_active=0., active_win_rate=0., n_days=len(rets))
        ann     = np.sqrt(252)
        sharpe  = rets.mean() / rets.std() * ann
        down    = rets[rets < 0].std()
        sortino = rets.mean() / down * ann if down > 0 else 0.
        cum     = (1 + rets).cumprod()
        max_dd  = ((cum - cum.cummax()) / cum.cummax()).min()
        ann_ret = (cum.iloc[-1] ** (252 / len(rets))) - 1
        calmar  = ann_ret / abs(max_dd) if max_dd != 0 else 0.

        # Time in position vs trade win rate (two separate metrics)
        # pct_active  = % of calendar days the strategy held a non-zero position
        # active_wr   = of those days, % that had a positive P&L (the real "win rate")
        active      = rets[rets.abs() > 1e-8]
        pct_active  = round(len(active) / len(rets), 3) if len(rets) > 0 else 0.
        active_wr   = round((active > 0).sum() / len(active), 3) if len(active) > 0 else 0.

        return dict(sharpe=round(sharpe, 3), sortino=round(sortino, 3),
                    max_drawdown=round(max_dd, 3),
                    ann_return=round(ann_ret, 3),
                    total_return=round(cum.iloc[-1] - 1, 3),
                    calmar=round(calmar, 3),
                    pct_active=pct_active,
                    active_win_rate=active_wr,
                    n_days=len(rets))


print("✅  PairTradingStrategy ready (with transaction costs + stop-loss)")

✅  PairTradingStrategy ready (with transaction costs + stop-loss)


---
## 🔄 Step 3 — Quarterly Walk-Forward Re-Optimization

This is the most important upgrade. Instead of optimizing once and applying forever, we:

1. Use the **last 2 years** of data as the BO training window
2. Run Optuna TPE (150 trials, 5-fold CV) to find the best parameters
3. Apply those parameters to the **next quarter only**
4. Slide the window forward 3 months and repeat

```
│ BO on 2016–2017 │─────── trade Q1 2018 ───────│
       │ BO on 2016.Q2–2018.Q1 │─── trade Q2 2018 ───│
              │ BO on 2016.Q3–2018.Q2 │── trade Q3 2018 ─│
                    ... and so on through 2025
```

This eliminates the core flaw of one-shot optimization: parameters always reflect
the most recent 2-year regime, not a stale 7-year average.

In [6]:
def make_objective(pa: pd.Series, pb: pd.Series, n_folds: int = N_CV_FOLDS):
    """
    Recency-weighted 5-fold time-series CV objective.
    Penalises negative folds; includes stop_z as 4th parameter.
    """
    n         = len(pa)
    fold_size = n // (n_folds + 1)
    folds     = [(f * fold_size, (f + 1) * fold_size) for f in range(n_folds)]
    weights   = np.arange(1, n_folds + 1, dtype=float)
    weights  /= weights.sum()

    def objective(trial: optuna.Trial) -> float:
        lookback = trial.suggest_int  ('lookback', 20,  120)
        entry_z  = trial.suggest_float('entry_z',  1.0, 3.5)
        exit_z   = trial.suggest_float('exit_z',   0.0, 1.0)
        stop_z   = trial.suggest_float('stop_z',   3.0, 6.0)

        if exit_z >= entry_z or entry_z >= stop_z:  # hard constraints
            return -99.0

        strat  = PairTradingStrategy(lookback=lookback, entry_z=entry_z,
                                      exit_z=exit_z, stop_z=stop_z)
        sharpes = []

        for start, end in folds:
            a, b = pa.iloc[start:end], pb.iloc[start:end]
            if len(a) < lookback + 20:
                sharpes.append(-99.)
                continue
            try:
                s = strat.metrics(strat.run(a, b)['returns'])['sharpe']
                sharpes.append(s * 1.5 if s < 0 else s)
            except Exception:
                sharpes.append(-99.)

        score = float(np.average(sharpes, weights=weights))
        if min(sharpes) < -3:
            score -= 3.
        return score

    return objective


print("✅  BO objective ready (4 parameters: lookback, entry_z, exit_z, stop_z)")

✅  BO objective ready (4 parameters: lookback, entry_z, exit_z, stop_z)


In [7]:
def run_walk_forward(pa: pd.Series, pb: pd.Series,
                     warmup_end: str    = WARMUP_END,
                     step_months: int   = WF_STEP,
                     train_years: int   = WF_TRAIN_YRS,
                     n_trials: int      = N_TRIALS,
                     verbose: bool      = True) -> dict:
    """
    Quarterly walk-forward BO.
    Returns full equity curve, parameter history, and per-period metrics.
    """
    all_returns  = []
    param_log    = []
    period_dates = []

    # Build walk-forward windows
    start_dt = pd.Timestamp(warmup_end) + pd.DateOffset(months=1)
    end_dt   = pa.index[-1]

    wf_starts = []
    current   = start_dt
    while current < end_dt:
        wf_starts.append(current)
        current += relativedelta(months=step_months)

    if verbose:
        print(f"Running {len(wf_starts)} walk-forward periods ({step_months}-month steps)...")
        print(f"Training window: {train_years} years of data per period\n")

    for i, period_start in enumerate(wf_starts):
        period_end = min(period_start + relativedelta(months=step_months) - pd.Timedelta(days=1),
                         end_dt)

        # Training window: [period_start - train_years, period_start - 1 day]
        train_end   = period_start - pd.Timedelta(days=1)
        train_start = train_end - relativedelta(years=train_years)

        pa_train = pa.loc[str(train_start.date()):str(train_end.date())]
        pb_train = pb.loc[str(train_start.date()):str(train_end.date())]
        pa_trade = pa.loc[str(period_start.date()):str(period_end.date())]
        pb_trade = pb.loc[str(period_start.date()):str(period_end.date())]

        if len(pa_train) < 60 or len(pa_trade) < 5:
            continue

        # Run BO on training window
        study = optuna.create_study(
            direction='maximize',
            sampler=optuna.samplers.TPESampler(seed=i),
            study_name=f"wf_{i}",
        )
        study.optimize(make_objective(pa_train, pb_train),
                       n_trials=n_trials, show_progress_bar=False)

        best = study.best_params
        strat = PairTradingStrategy(**best)
        result = strat.run(pa_trade, pb_trade)

        all_returns.append(result['returns'])
        param_log.append({
            'period_start': period_start,
            'period_end':   period_end,
            'train_start':  train_start,
            'train_end':    train_end,
            **best,
            'cv_sharpe':    round(study.best_value, 3),
            'n_trades':     result['n_trades'],
        })
        period_dates.append((period_start, period_end))

        if verbose and ((i + 1) % 4 == 0 or i == 0):
            print(f"  Period {i+1:>3}/{len(wf_starts)}  {period_start.date()} → {period_end.date()}"
                  f"  | best: lookback={best['lookback']}, entry_z={best['entry_z']:.2f}"
                  f", stop_z={best['stop_z']:.2f}  CV Sharpe={study.best_value:.3f}")

    full_returns = pd.concat(all_returns).sort_index() if all_returns else pd.Series(dtype=float)
    param_df     = pd.DataFrame(param_log)

    return {'returns': full_returns, 'params': param_df, 'periods': period_dates}


print("✅  Walk-forward engine ready")

✅  Walk-forward engine ready


In [8]:
# ── Select top pairs from warmup cointegration screen ────────────────────────
top_pairs = coint_pairs.head(TOP_N_PAIRS).copy()

print("Selected pairs for walk-forward:")
for _, r in top_pairs.iterrows():
    both = "✅ both tests" if r['passes_both'] else "⚠️ one test only"
    print(f"  {r['label']:<12} EG p={r['eg_pvalue']:.4f}  Jo p={r['jo_pvalue']:.2f}"
          f"  hl={r['half_life']}d  {both}")
print()


# ── Worker function (runs one pair; picklable for joblib) ─────────────────────
def _run_pair(row, prices_df):
    t1, t2 = row['ticker_1'], row['ticker_2']
    label  = row['label']
    if t1 not in prices_df.columns or t2 not in prices_df.columns:
        return None

    pa = prices_df[t1].loc[FULL_START:FULL_END]
    pb = prices_df[t2].loc[FULL_START:FULL_END]

    wf        = run_walk_forward(pa, pb, verbose=False)
    oos_start = pd.Timestamp(WARMUP_END) + pd.DateOffset(months=1)
    strat_b   = PairTradingStrategy(**BASELINE_PARAMS)
    res_b     = strat_b.run(pa.loc[oos_start:], pb.loc[oos_start:])

    return (label,
            {**wf, 'pair': (t1, t2)},
            {'returns': res_b['returns'], 'pair': (t1, t2)},
            PairTradingStrategy.metrics(wf['returns']),
            PairTradingStrategy.metrics(res_b['returns']))


# ── Execute: parallel (USE_PARALLEL=True) or sequential ──────────────────────
n_workers = min(len(top_pairs), 4)
if USE_PARALLEL:
    print(f"Running {len(top_pairs)} pairs in parallel (n_jobs={n_workers})...")
    raw_results = Parallel(n_jobs=n_workers, backend='threading')(
        delayed(_run_pair)(row, prices) for _, row in top_pairs.iterrows()
    )
else:
    raw_results = [_run_pair(row, prices) for _, row in top_pairs.iterrows()]

# ── Collect results & print per-pair comparison ──────────────────────────────
wf_results   = {}
base_results = {}

for res in raw_results:
    if res is None:
        continue
    label, wf, base, m_wf, m_base = res
    wf_results[label]   = wf
    base_results[label] = base

    print(f"\n{'='*60}")
    print(f"📊  {label}  —  Walk-Fwd BO vs Baseline")
    print('='*60)
    print(f"  {'Metric':<18} {'Baseline':>10} {'Walk-Fwd BO':>12}  {'Δ':>8}")
    print("  " + "-" * 52)
    for metric in ['sharpe', 'sortino', 'max_drawdown', 'total_return', 'calmar']:
        bv, wv = m_base[metric], m_wf[metric]
        try:
            d  = wv - bv
            sg = "▲" if d > 0 else "▼"
            print(f"  {metric:<18} {bv:>10.3f} {wv:>12.3f}  {sg}{abs(d):>6.3f}")
        except Exception:
            print(f"  {metric:<18} {bv:>10.3f} {wv:>12.3f}")

print("\n\n✅  All pairs complete")

Selected pairs for walk-forward:
  META/NVDA    EG p=0.0881  Jo p=0.01  hl=27.0d  ✅ both tests
  GS/WFC       EG p=0.0555  Jo p=0.15  hl=31.8d  ⚠️ one test only
  GOOGL/NVDA   EG p=0.0582  Jo p=0.15  hl=24.6d  ⚠️ one test only
  MDT/PFE      EG p=0.0818  Jo p=0.15  hl=22.8d  ⚠️ one test only
  PEP/YUM      EG p=0.0902  Jo p=0.15  hl=37.7d  ⚠️ one test only

Running 5 pairs in parallel (n_jobs=4)...



📊  META/NVDA  —  Walk-Fwd BO vs Baseline
  Metric               Baseline  Walk-Fwd BO         Δ
  ----------------------------------------------------
  sharpe                 -0.688        0.236  ▲ 0.924
  sortino                -0.588        0.126  ▲ 0.714
  max_drawdown           -0.785       -0.101  ▲ 0.684
  total_return           -0.692        0.102  ▲ 0.794
  calmar                 -0.200        0.140  ▲ 0.340

📊  GS/WFC  —  Walk-Fwd BO vs Baseline
  Metric               Baseline  Walk-Fwd BO         Δ
  ----------------------------------------------------
  sharpe                  0.470        0.740  ▲ 0.270
  sortino                 0.540        0.400  ▼ 0.140
  max_drawdown           -0.201       -0.081  ▲ 0.120
  total_return            0.356        0.302  ▼ 0.054
  calmar                  0.224        0.480  ▲ 0.256

📊  GOOGL/NVDA  —  Walk-Fwd BO vs Baseline
  Metric               Baseline  Walk-Fwd BO         Δ
  ----------------------------------------------------
  shar

---
## 📊 Step 4 — Interactive Plotly Dashboard

Five panels per pair:
1. **Cumulative Returns** — Walk-forward BO vs fixed baseline (with regime bands)
2. **Parameter Evolution** — How lookback, entry_z, stop_z change each quarter
3. **Rolling Sharpe (6M)** — Strategy stability over time
4. **Drawdown Comparison** — Underwater equity curve
5. **Trade Frequency** — Number of trades per period

In [9]:
REGIME_BANDS = [
    ('2016-01-01', '2019-12-31', 'rgba(100,200,100,0.06)', 'Bull Run'),
    ('2020-01-01', '2020-12-31', 'rgba(200,100,100,0.10)', 'COVID Crash'),
    ('2021-01-01', '2022-12-31', 'rgba(200,150,50,0.07)',  'Rate Hikes'),
    ('2023-01-01', '2025-12-31', 'rgba(100,150,250,0.06)', 'AI Rally'),
]


def build_dashboard(label: str) -> go.Figure:
    wf      = wf_results[label]
    base    = base_results[label]
    params  = wf['params']

    ret_wf   = wf['returns']
    ret_base = base['returns'].reindex(ret_wf.index).fillna(0)

    cum_wf   = (1 + ret_wf).cumprod()
    cum_base = (1 + ret_base).cumprod()

    # Rolling Sharpe
    roll_sharpe_wf   = ret_wf.rolling(126).apply(
        lambda r: r.mean() / r.std() * np.sqrt(252) if r.std() > 0 else 0)
    roll_sharpe_base = ret_base.rolling(126).apply(
        lambda r: r.mean() / r.std() * np.sqrt(252) if r.std() > 0 else 0)

    # Drawdowns
    dd_wf   = (cum_wf   - cum_wf.cummax())   / cum_wf.cummax()
    dd_base = (cum_base - cum_base.cummax()) / cum_base.cummax()

    fig = make_subplots(
        rows=3, cols=2,
        subplot_titles=[
            f"1️⃣  Cumulative Returns — {label}",
            f"2️⃣  Parameter Evolution (Quarterly BO)",
            f"3️⃣  Rolling 6-Month Sharpe",
            f"4️⃣  Drawdown Comparison",
            f"5️⃣  Trade Frequency per Period",
            f"6️⃣  lookback Distribution (BO Trials)",
        ],
        vertical_spacing=0.13, horizontal_spacing=0.10,
        specs=[[{}, {}], [{}, {}], [{}, {}]],
    )

    # Helper: add regime bands to a panel
    def add_regimes(row, col):
        for s, e, col_r, name in REGIME_BANDS:
            fig.add_vrect(x0=s, x1=e, fillcolor=col_r, layer='below',
                          line_width=0, row=row, col=col,
                          annotation_text=name, annotation_position='top left',
                          annotation_font_size=8, annotation_font_color='rgba(200,200,200,0.6)')

    # ── Panel 1: Cumulative returns ──
    add_regimes(1, 1)
    fig.add_trace(go.Scatter(x=cum_base.index, y=cum_base, name='Baseline (fixed)',
                             line=dict(color='#636EFA', dash='dash', width=2)), row=1, col=1)
    fig.add_trace(go.Scatter(x=cum_wf.index,   y=cum_wf,   name='Walk-Fwd BO',
                             line=dict(color='#EF553B', width=2.5)), row=1, col=1)

    # ── Panel 2: Parameter evolution ──
    if not params.empty:
        for param, color, name in [
            ('lookback', '#00CC96', 'lookback (days)'),
            ('entry_z',  '#FFA15A', 'entry_z (×10)'),
            ('stop_z',   '#AB63FA', 'stop_z'),
        ]:
            if param not in params.columns:
                continue
            y_vals = params[param] * 10 if param == 'entry_z' else params[param]
            fig.add_trace(go.Scatter(
                x=params['period_start'], y=y_vals, name=name, mode='lines+markers',
                line=dict(color=color, width=1.8),
                marker=dict(size=5),
            ), row=1, col=2)

    # ── Panel 3: Rolling Sharpe ──
    add_regimes(2, 1)
    fig.add_trace(go.Scatter(x=roll_sharpe_base.index, y=roll_sharpe_base,
                             name='Baseline', line=dict(color='#636EFA', dash='dash', width=1.5)),
                  row=2, col=1)
    fig.add_trace(go.Scatter(x=roll_sharpe_wf.index, y=roll_sharpe_wf,
                             name='Walk-Fwd', line=dict(color='#EF553B', width=2)),
                  row=2, col=1)
    fig.add_hline(y=0, line_color='rgba(200,200,200,0.3)', line_dash='dot', row=2, col=1)
    fig.add_hline(y=1, line_color='rgba(100,200,100,0.4)', line_dash='dot', row=2, col=1)

    # ── Panel 4: Drawdowns ──
    add_regimes(2, 2)
    fig.add_trace(go.Scatter(x=dd_base.index, y=dd_base * 100,
                             name='DD Baseline', line=dict(color='#636EFA', dash='dash'),
                             fill='tozeroy', fillcolor='rgba(99,110,250,0.1)'), row=2, col=2)
    fig.add_trace(go.Scatter(x=dd_wf.index, y=dd_wf * 100,
                             name='DD Walk-Fwd', line=dict(color='#EF553B'),
                             fill='tozeroy', fillcolor='rgba(239,85,59,0.1)'), row=2, col=2)

    # ── Panel 5: Trade frequency ──
    if not params.empty and 'n_trades' in params.columns:
        fig.add_trace(go.Bar(
            x=params['period_start'].astype(str),
            y=params['n_trades'],
            name='Trades/period',
            marker_color='#19D3F3',
            opacity=0.8,
        ), row=3, col=1)

    # ── Panel 6: Lookback histogram ──
    if not params.empty and 'lookback' in params.columns:
        fig.add_trace(go.Histogram(
            x=params['lookback'], nbinsx=15,
            name='lookback dist',
            marker_color='#00CC96', opacity=0.75,
        ), row=3, col=2)

    # ── Layout ──
    fig.update_layout(
        height=900, template='plotly_dark',
        title=dict(
            text=f"<b>Walk-Forward Bayesian Optimization — {label} (2016–2025)</b>",
            font=dict(size=15),
        ),
        legend=dict(orientation='h', y=-0.08, x=0.5, xanchor='center', font=dict(size=9)),
        margin=dict(t=90, b=100),
    )
    fig.update_yaxes(title_text='Cum. Return',   row=1, col=1)
    fig.update_yaxes(title_text='Param Value',   row=1, col=2)
    fig.update_yaxes(title_text='Rolling Sharpe',row=2, col=1)
    fig.update_yaxes(title_text='Drawdown (%)',  row=2, col=2)
    fig.update_yaxes(title_text='# Trades',      row=3, col=1)
    fig.update_yaxes(title_text='Frequency',     row=3, col=2)
    fig.update_xaxes(title_text='Quarter',        row=3, col=1)
    fig.update_xaxes(title_text='Lookback (days)',row=3, col=2)

    return fig


for lbl in list(wf_results.keys()):
    build_dashboard(lbl).show()
    print(f"\n📊  Dashboard: {lbl}")

# ── Save all dashboards (HTML + PNG) ──────────────────────────────────────────
print("\nSaving dashboards to disk...")
for lbl in list(wf_results.keys()):
    fig      = build_dashboard(lbl)
    safe_lbl = lbl.replace('/', '_')

    html_path = f"{OUTPUT_DIR}/charts/{safe_lbl}_dashboard.html"
    fig.write_html(html_path)

    png_path = f"{OUTPUT_DIR}/charts/{safe_lbl}_dashboard.png"
    try:
        fig.write_image(png_path, width=1600, height=900, scale=2)
        print(f"  ✅  {lbl}  →  HTML + PNG saved")
    except Exception as e:
        print(f"  ⚠️  {lbl}  →  HTML saved (PNG skipped: {e})")

print(f"\n📁  Charts saved → ./{OUTPUT_DIR}/charts/")


📊  Dashboard: META/NVDA



📊  Dashboard: GS/WFC



📊  Dashboard: GOOGL/NVDA



📊  Dashboard: MDT/PFE



📊  Dashboard: PEP/YUM

Saving dashboards to disk...


  ✅  META/NVDA  →  HTML + PNG saved


  ✅  GS/WFC  →  HTML + PNG saved


  ✅  GOOGL/NVDA  →  HTML + PNG saved


  ✅  MDT/PFE  →  HTML + PNG saved


  ✅  PEP/YUM  →  HTML + PNG saved

📁  Charts saved → ./outputs/charts/


---
## ✅ Final Summary

In [10]:
# ================================================================
# FINAL SUMMARY
# ================================================================
print("\n" + "=" * 78)
print("FINAL RESULTS — WALK-FORWARD (2019–2025) vs FIXED BASELINE")
print("=" * 78)
print(f"{'Pair':<12} {'Method':<15} {'Sharpe':>7} {'Sortino':>8} {'AnnRet':>8}"
      f" {'MaxDD':>8} {'Calmar':>7} {'%Active':>8} {'WinRate*':>9}")
print("-" * 92)
print("  * WinRate = % of days IN POSITION with positive P&L (not % of all calendar days)")
print()

for label in wf_results:
    m_wf   = PairTradingStrategy.metrics(wf_results[label]['returns'])
    m_base = PairTradingStrategy.metrics(base_results[label]['returns'])

    for m, name in [(m_base, 'Baseline (fixed)'), (m_wf, 'Walk-Fwd BO')]:
        flag = " ◀" if name == 'Walk-Fwd BO' and m['sharpe'] > m_base['sharpe'] else ""
        print(f"{label:<12} {name:<15} {m['sharpe']:>7.3f} {m['sortino']:>8.3f}"
              f" {m['ann_return']:>8.1%} {m['max_drawdown']:>8.1%}"
              f" {m['calmar']:>7.3f} {m['pct_active']:>8.1%} {m['active_win_rate']:>9.1%}{flag}")
    print()

print("=" * 78)
print("\n⚠️  Transaction costs: 2 bps/leg applied throughout.")
print("⚠️  Past performance does not guarantee future results.")
print("⚠️  Universe of 42 S&P 500 stocks — not a fully representative cross-section.")

# ── Export metrics & parameters to CSV ────────────────────────────────────────
rows = []
for label in wf_results:
    m_wf   = PairTradingStrategy.metrics(wf_results[label]['returns'])
    m_base = PairTradingStrategy.metrics(base_results[label]['returns'])
    for m, method in [(m_base, 'Baseline (fixed)'), (m_wf, 'Walk-Fwd BO')]:
        rows.append({'pair': label, 'method': method, **m})

summary_df   = pd.DataFrame(rows)
summary_path = f"{OUTPUT_DIR}/data/results_summary.csv"
summary_df.to_csv(summary_path, index=False)
print(f"\n✅  Metrics CSV  → ./{summary_path}")

for label, wf in wf_results.items():
    if not wf['params'].empty:
        safe = label.replace('/', '_')
        path = f"{OUTPUT_DIR}/data/{safe}_params.csv"
        wf['params'].to_csv(path, index=False)
        print(f"✅  Params  CSV  → ./{path}")

print(f"\n📁  All data saved → ./{OUTPUT_DIR}/data/")
display(summary_df)


FINAL RESULTS — WALK-FORWARD (2019–2025) vs FIXED BASELINE
Pair         Method           Sharpe  Sortino   AnnRet    MaxDD  Calmar  %Active  WinRate*
--------------------------------------------------------------------------------------------
  * WinRate = % of days IN POSITION with positive P&L (not % of all calendar days)

META/NVDA    Baseline (fixed)  -0.688   -0.588   -15.7%   -78.5%  -0.200    53.1%     48.6%
META/NVDA    Walk-Fwd BO       0.236    0.126     1.4%   -10.1%   0.140    10.3%     42.5% ◀

GS/WFC       Baseline (fixed)   0.470    0.540     4.5%   -20.1%   0.224    49.9%     50.0%
GS/WFC       Walk-Fwd BO       0.740    0.400     3.9%    -8.1%   0.480    10.9%     47.4% ◀

GOOGL/NVDA   Baseline (fixed)  -0.137   -0.122    -3.0%   -36.6%  -0.083    48.1%     49.0%
GOOGL/NVDA   Walk-Fwd BO      -0.129   -0.057    -0.9%   -13.0%  -0.069     8.5%     39.2% ◀

MDT/PFE      Baseline (fixed)  -0.086   -0.089    -1.6%   -26.9%  -0.059    44.0%     46.5%
MDT/PFE      Walk-Fwd 

,pair,method,sharpe,sortino,max_drawdown,ann_return,total_return,calmar,pct_active,active_win_rate,n_days
0,META/NVDA,Baseline (fixed),-0.688,-0.588,-0.785,-0.157,-0.692,-0.200,0.531,0.486,1739
1,META/NVDA,Walk-Fwd BO,0.236,0.126,-0.101,0.014,0.102,0.140,0.103,0.425,1739
2,GS/WFC,Baseline (fixed),0.470,0.540,-0.201,0.045,0.356,0.224,0.499,0.500,1739
3,GS/WFC,Walk-Fwd BO,0.740,0.400,-0.081,0.039,0.302,0.480,0.109,0.474,1739
4,GOOGL/NVDA,Baseline (fixed),-0.137,-0.122,-0.366,-0.030,-0.191,-0.083,0.481,0.490,1739
5,GOOGL/NVDA,Walk-Fwd BO,-0.129,-0.057,-0.130,-0.009,-0.061,-0.069,0.085,0.392,1739
6,MDT/PFE,Baseline (fixed),-0.086,-0.089,-0.269,-0.016,-0.105,-0.059,0.440,0.465,1739
7,MDT/PFE,Walk-Fwd BO,-0.456,-0.159,-0.134,-0.020,-0.131,-0.150,0.066,0.386,1739
8,PEP/YUM,Baseline (fixed),-0.383,-0.301,-0.269,-0.034,-0.212,-0.126,0.409,0.463,1739
9,PEP/YUM,Walk-Fwd BO,-0.083,-0.036,-0.118,-0.004,-0.025,-0.031,0.087,0.421,1739


---
## 🧩 Step 5 — Adaptive Portfolio: Annual Pair Re-Selection + SPY Benchmark

Two upgrades combined:

**Annual pair re-selection:** Cointegration is re-run at the start of each calendar year using
the most recent 2-year training window. The top pairs at that moment replace the previous year's
selection — so the universe adapts to regime changes (e.g. energy cointegration strengthening
during 2021–2022 rate hikes, then weakening post-2023).

**Equal-weight portfolio:** The top 2 pairs selected each year are traded simultaneously with
equal capital weight. Returns are averaged across the active pairs each day, giving a smoother
equity curve than any single pair.

**SPY benchmark:** Buy-and-hold SPY over the same OOS period provides market context — a
market-neutral strategy should show low correlation with SPY while generating positive alpha.

In [11]:
def run_adaptive_portfolio(all_prices, warmup_end=WARMUP_END,
                           step_months=WF_STEP, train_years=WF_TRAIN_YRS,
                           n_trials=N_ADAPTIVE_TRIALS, top_n=TOP_N_ADAPTIVE):
    """
    Annual pair re-selection + quarterly walk-forward BO.
    At each calendar year boundary, re-run cointegration on the training window
    to select fresh pairs. Trade the top_n pairs with equal weight each quarter.
    """
    all_rets      = []
    selection_log = []

    start_dt  = pd.Timestamp(warmup_end) + pd.DateOffset(months=1)
    end_dt    = all_prices.index[-1]
    wf_starts = []
    current   = start_dt
    while current < end_dt:
        wf_starts.append(current)
        current += relativedelta(months=step_months)

    active_pairs = []
    last_year    = None

    print(f"Adaptive portfolio: {len(wf_starts)} periods, annual re-selection, top-{top_n} pairs")
    print(f"BO trials per period per pair: {n_trials}\n")

    for i, period_start in enumerate(wf_starts):
        period_end  = min(period_start + relativedelta(months=step_months) - pd.Timedelta(days=1), end_dt)
        train_end   = period_start - pd.Timedelta(days=1)
        train_start = train_end - relativedelta(years=train_years)

        train_slice = all_prices.loc[str(train_start.date()):str(train_end.date())]

        # ── Annual pair re-selection ──────────────────────────────────────────
        if last_year != period_start.year:
            new_coint = find_cointegrated_pairs(train_slice, sector_only=SECTOR_ONLY_PAIRS)
            if not new_coint.empty:
                active_pairs = [(r['ticker_1'], r['ticker_2'])
                                for _, r in new_coint.head(top_n).iterrows()
                                if r['ticker_1'] in all_prices.columns
                                   and r['ticker_2'] in all_prices.columns]
            last_year = period_start.year
            pair_labels = [f"{t1}/{t2}" for t1, t2 in active_pairs]
            selection_log.append({'year': period_start.year, 'pairs': ', '.join(pair_labels)})
            print(f"  {period_start.year}  re-selected: {pair_labels}")

        if not active_pairs:
            continue

        # ── BO + trade each active pair ───────────────────────────────────────
        period_rets = []
        for t1, t2 in active_pairs:
            pa_tr = train_slice[t1] if t1 in train_slice.columns else pd.Series()
            pb_tr = train_slice[t2] if t2 in train_slice.columns else pd.Series()
            pa_td = all_prices[t1].loc[str(period_start.date()):str(period_end.date())]
            pb_td = all_prices[t2].loc[str(period_start.date()):str(period_end.date())]

            if len(pa_tr) < 60 or len(pa_td) < 5:
                continue

            study = optuna.create_study(
                direction='maximize',
                sampler=optuna.samplers.TPESampler(seed=i * 100 + len(t1)),
            )
            study.optimize(make_objective(pa_tr, pb_tr),
                           n_trials=n_trials, show_progress_bar=False)

            strat  = PairTradingStrategy(**study.best_params)
            result = strat.run(pa_td, pb_td)
            period_rets.append(result['returns'])

        if period_rets:
            # Equal-weight portfolio return for this quarter
            port_quarter = pd.concat(period_rets, axis=1).mean(axis=1)
            all_rets.append(port_quarter)

    port_returns = pd.concat(all_rets).sort_index() if all_rets else pd.Series(dtype=float)
    sel_df       = pd.DataFrame(selection_log)
    return {'returns': port_returns, 'selection_log': sel_df}


# ── Run adaptive portfolio ────────────────────────────────────────────────────
print("=" * 60)
print("ADAPTIVE PORTFOLIO — Annual Re-Selection")
print("=" * 60)
adaptive = run_adaptive_portfolio(prices)

# ── Download SPY as benchmark ─────────────────────────────────────────────────
oos_start = pd.Timestamp(WARMUP_END) + pd.DateOffset(months=1)
spy_raw   = yf.download('SPY', start=str(oos_start.date()), end=FULL_END,
                         auto_adjust=True, progress=False)
spy_ret   = spy_raw['Close'].squeeze().pct_change().dropna()

# ── Align returns ────────────────────────────────────────────────────────────
port_ret = adaptive['returns']
idx      = port_ret.index.intersection(spy_ret.index)
port_ret = port_ret.loc[idx].fillna(0)
spy_ret  = spy_ret.loc[idx].fillna(0)

cum_port = (1 + port_ret).cumprod()
cum_spy  = (1 + spy_ret).cumprod()

# ── Metrics ──────────────────────────────────────────────────────────────────
m_port = PairTradingStrategy.metrics(port_ret)
m_spy  = PairTradingStrategy.metrics(spy_ret)

# ── Market-neutrality metrics ─────────────────────────────────────────────────
beta_to_spy = np.cov(port_ret.values, spy_ret.values)[0, 1] / spy_ret.var()
corr_to_spy = np.corrcoef(port_ret.values, spy_ret.values)[0, 1]
roll_beta   = (port_ret.rolling(63)
               .cov(spy_ret)
               .divide(spy_ret.rolling(63).var())
               .rename('rolling_beta'))

print(f"\n{'Metric':<22} {'Adaptive Port':>14} {'Buy-Hold SPY':>14}")
print("-" * 52)
for metric in ['sharpe', 'sortino', 'ann_return', 'max_drawdown', 'calmar']:
    print(f"{metric:<22} {m_port[metric]:>14.3f} {m_spy[metric]:>14.3f}")
print()
print(f"{'beta to SPY':<22} {beta_to_spy:>14.3f} {'1.000':>14}")
print(f"{'corr to SPY':<22} {corr_to_spy:>14.3f} {'1.000':>14}")
print(f"{'% days active':<22} {m_port['pct_active']:>14.1%}")
print(f"{'active win rate':<22} {m_port['active_win_rate']:>14.1%}")
print()
print("📌 Interpretation:")
print(f"   Beta ≈ {beta_to_spy:.2f} confirms near-zero market exposure (by construction: long A, short B).")
print(f"   Corr ≈ {corr_to_spy:.2f} — portfolio returns are independent of SPY direction.")
print(f"   SPY +{m_spy['ann_return']:.1%} annualized is irrelevant as a benchmark;")
print(f"   the correct benchmark is 0% (cash). Portfolio delivered {m_port['ann_return']:.1%} annualized.")

# ── Plot: Portfolio vs SPY ─────────────────────────────────────────────────────
fig_port = make_subplots(rows=3, cols=1,
    subplot_titles=['① Cumulative Return — Adaptive Portfolio vs SPY (note: different benchmarks)',
                    '② Drawdown — Adaptive Portfolio vs SPY',
                    f'③ Rolling 63-day Beta to SPY  (target: ≈ 0.0, actual avg: {beta_to_spy:.2f})'],
    vertical_spacing=0.10,
    row_heights=[0.40, 0.30, 0.30])

for s, e, col_r, name in REGIME_BANDS:
    for row in [1, 2, 3]:
        fig_port.add_vrect(x0=s, x1=e, fillcolor=col_r, layer='below',
                           line_width=0, row=row, col=1,
                           annotation_text=name if row == 1 else '',
                           annotation_position='top left',
                           annotation_font_size=8,
                           annotation_font_color='rgba(200,200,200,0.6)')

fig_port.add_trace(go.Scatter(x=cum_spy.index,  y=cum_spy,  name='Buy-Hold SPY',
                               line=dict(color='#636EFA', dash='dash', width=2)), row=1, col=1)
fig_port.add_trace(go.Scatter(x=cum_port.index, y=cum_port, name='Adaptive Portfolio',
                               line=dict(color='#EF553B', width=2.5)), row=1, col=1)

dd_port = (cum_port - cum_port.cummax()) / cum_port.cummax()
dd_spy  = (cum_spy  - cum_spy.cummax())  / cum_spy.cummax()
fig_port.add_trace(go.Scatter(x=dd_spy.index,  y=dd_spy * 100,  name='SPY DD',
                               line=dict(color='#636EFA', dash='dash'),
                               fill='tozeroy', fillcolor='rgba(99,110,250,0.08)'), row=2, col=1)
fig_port.add_trace(go.Scatter(x=dd_port.index, y=dd_port * 100, name='Portfolio DD',
                               line=dict(color='#EF553B'),
                               fill='tozeroy', fillcolor='rgba(239,85,59,0.1)'),  row=2, col=1)

# Rolling beta panel
fig_port.add_trace(go.Scatter(
    x=roll_beta.index, y=roll_beta.values,
    name='Rolling Beta (63d)',
    line=dict(color='#FFD700', width=1.8),
    fill='tozeroy', fillcolor='rgba(255,215,0,0.08)',
), row=3, col=1)
fig_port.add_hline(y=0,    line_color='rgba(200,200,200,0.4)', line_dash='dot', row=3, col=1)
fig_port.add_hline(y=0.2,  line_color='rgba(200,100,100,0.3)', line_dash='dot', row=3, col=1)
fig_port.add_hline(y=-0.2, line_color='rgba(200,100,100,0.3)', line_dash='dot', row=3, col=1)
fig_port.add_annotation(x=roll_beta.index[-1], y=0.25, text='±0.2 band',
                         font=dict(size=8, color='rgba(200,100,100,0.7)'),
                         showarrow=False, row=3, col=1)

fig_port.update_layout(height=850, template='plotly_dark',
    title='<b>Adaptive Portfolio (Annual Re-Selection) vs SPY  |  '
          f'β={beta_to_spy:.2f}  ρ={corr_to_spy:.2f}</b>',
    legend=dict(orientation='h', y=-0.06, x=0.5, xanchor='center'))
fig_port.update_yaxes(title_text='Cum. Return',  row=1, col=1)
fig_port.update_yaxes(title_text='Drawdown (%)', row=2, col=1)
fig_port.update_yaxes(title_text='Beta to SPY',  row=3, col=1)
fig_port.show()

# ── Save ─────────────────────────────────────────────────────────────────────
fig_port.write_html(f"{OUTPUT_DIR}/charts/adaptive_portfolio_vs_spy.html")
try:
    fig_port.write_image(f"{OUTPUT_DIR}/charts/adaptive_portfolio_vs_spy.png",
                          width=1400, height=650, scale=2)
    print(f"\n✅  Portfolio chart saved (HTML + PNG)")
except Exception as e:
    print(f"\n⚠️  PNG save skipped: {e}")

print("\n🗓️  Annual pair selection log:")
display(adaptive['selection_log'])

ADAPTIVE PORTFOLIO — Annual Re-Selection
Adaptive portfolio: 28 periods, annual re-selection, top-2 pairs
BO trials per period per pair: 50

Testing 126 within-sector pairs (EG + Johansen, threshold p < 0.1)...



✅  17 pairs pass at least one test  (4 pass both EG + Johansen)
  2019  re-selected: ['OXY/PSX', 'CL/NKE']


Testing 126 within-sector pairs (EG + Johansen, threshold p < 0.1)...



✅  23 pairs pass at least one test  (8 pass both EG + Johansen)
  2020  re-selected: ['CAT/GE', 'JNJ/LLY']


Testing 126 within-sector pairs (EG + Johansen, threshold p < 0.1)...


...

Testing 126 within-sector pairs (EG + Johansen, threshold p < 0.1)...



✅  20 pairs pass at least one test  (4 pass both EG + Johansen)
  2025  re-selected: ['CVX/PSX', 'CVX/SLB']



Metric                  Adaptive Port   Buy-Hold SPY
----------------------------------------------------
sharpe                         -0.269          0.861
sortino                        -0.118          1.048
ann_return                     -0.011          0.163
max_drawdown                   -0.107         -0.337
calmar                         -0.099          0.483

beta to SPY                    -0.010          1.000
corr to SPY                    -0.052          1.000
% days active                   10.5%
active win rate                 35.2%

📌 Interpretation:
   Beta ≈ -0.01 confirms near-zero market exposure (by construction: long A, short B).
   Corr ≈ -0.05 — portfolio returns are independent of SPY direction.
   SPY +16.3% annualized is irrelevant as a benchmark;
   the correct benchmark is 0% (cash). Portfolio delivered -1.1% annualized.



✅  Portfolio chart saved (HTML + PNG)

🗓️  Annual pair selection log:


,year,pairs
0,2019,"OXY/PSX, CL/NKE"
1,2020,"CAT/GE, JNJ/LLY"
2,2021,"EOG/XOM, EOG/OXY"
3,2022,"OXY/XOM, GOOGL/ORCL"
4,2023,"BA/GE, BMY/JNJ"
5,2024,"META/NVDA, OXY/SLB"
6,2025,"CVX/PSX, CVX/SLB"


---
## 📊 Step 6 — Per-Pair Attribution Analysis

Which pairs drove performance — and in which market regimes?

- **Regime Sharpe contribution:** the rolling Sharpe each pair achieved within each macro regime
- **Trade frequency heatmap:** shows where BO found tradeable signals vs. where it went quiet
- **Correlation matrix:** confirms the pairs are sufficiently independent (low pairwise return correlation = good portfolio diversification)

In [12]:
# ================================================================
# ATTRIBUTION ANALYSIS
# ================================================================

REGIME_WINDOWS = {
    'Bull Run\n2019':    ('2019-01-01', '2019-12-31'),
    'COVID\n2020':       ('2020-01-01', '2020-12-31'),
    'Rate Hikes\n21–22': ('2021-01-01', '2022-12-31'),
    'AI Rally\n23–25':   ('2023-01-01', '2025-12-31'),
}

pairs   = list(wf_results.keys())
regimes = list(REGIME_WINDOWS.keys())

# ── 1. Regime Sharpe matrix ──────────────────────────────────────────────────
sharpe_matrix = pd.DataFrame(index=pairs, columns=regimes, dtype=float)
for pair in pairs:
    rets = wf_results[pair]['returns']
    for regime, (s, e) in REGIME_WINDOWS.items():
        slice_ = rets.loc[s:e].dropna()
        if len(slice_) > 10 and slice_.std() > 0:
            sharpe_matrix.loc[pair, regime] = round(
                slice_.mean() / slice_.std() * np.sqrt(252), 2)
        else:
            sharpe_matrix.loc[pair, regime] = 0.0

fig_attr = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        '① Regime Sharpe (Walk-Fwd BO per pair)',
        '② Trades per Regime',
        '③ Pairwise Return Correlation',
    ],
    horizontal_spacing=0.12,
)

# Heatmap: regime Sharpe
z_vals  = sharpe_matrix.values.astype(float)
z_text  = [[f"{v:.2f}" for v in row] for row in z_vals]
fig_attr.add_trace(go.Heatmap(
    z=z_vals, x=regimes, y=pairs,
    text=z_text, texttemplate='%{text}',
    colorscale='RdYlGn', zmid=0,
    showscale=True, colorbar=dict(x=0.28, len=0.9, title='Sharpe'),
), row=1, col=1)

# ── 2. Trades-per-regime bar ─────────────────────────────────────────────────
trade_matrix = pd.DataFrame(index=pairs, columns=regimes, dtype=float)
for pair in pairs:
    params = wf_results[pair]['params']
    if params.empty or 'n_trades' not in params.columns:
        trade_matrix.loc[pair] = 0
        continue
    for regime, (s, e) in REGIME_WINDOWS.items():
        mask = (params['period_start'] >= s) & (params['period_start'] <= e)
        trade_matrix.loc[pair, regime] = params.loc[mask, 'n_trades'].sum()

for j, regime in enumerate(regimes):
    fig_attr.add_trace(go.Bar(
        name=regime.replace('\n', ' '),
        x=pairs,
        y=trade_matrix[regime].values.astype(float),
        text=trade_matrix[regime].values.astype(int),
        textposition='auto',
        showlegend=(j == 0),
    ), row=1, col=2)

# ── 3. Correlation heatmap ───────────────────────────────────────────────────
ret_df = pd.DataFrame({p: wf_results[p]['returns'] for p in pairs}).fillna(0)
corr   = ret_df.corr().round(2)
fig_attr.add_trace(go.Heatmap(
    z=corr.values, x=corr.columns.tolist(), y=corr.index.tolist(),
    text=corr.values.round(2),
    texttemplate='%{text}',
    colorscale='RdBu_r', zmid=0, zmin=-1, zmax=1,
    showscale=True, colorbar=dict(x=1.01, len=0.9, title='Corr'),
), row=1, col=3)

fig_attr.update_layout(
    height=480, template='plotly_dark',
    title='<b>Attribution Analysis — Per-Pair, Per-Regime</b>',
    barmode='group',
    legend=dict(orientation='h', y=-0.18, x=0.5, xanchor='center', font=dict(size=8)),
    margin=dict(t=80, b=100),
)
fig_attr.show()

# ── Save ─────────────────────────────────────────────────────────────────────
fig_attr.write_html(f"{OUTPUT_DIR}/charts/attribution_analysis.html")
try:
    fig_attr.write_image(f"{OUTPUT_DIR}/charts/attribution_analysis.png",
                          width=1600, height=480, scale=2)
    print("✅  Attribution chart saved (HTML + PNG)")
except Exception as e:
    print(f"⚠️  PNG save skipped: {e}")

print("\n📊  Regime Sharpe matrix:")
display(sharpe_matrix)
print("\n🔗  Pairwise correlation:")
display(corr)

✅  Attribution chart saved (HTML + PNG)

📊  Regime Sharpe matrix:


,Bull Run\n2019,COVID\n2020,Rate Hikes\n21–22,AI Rally\n23–25
META/NVDA,1.60,-1.01,0.49,-0.03
GS/WFC,-0.09,2.45,0.80,0.13
GOOGL/NVDA,-0.10,0.21,-0.58,0.03
MDT/PFE,-1.45,0.07,-0.31,-0.66
PEP/YUM,0.71,-1.42,-0.13,0.19



🔗  Pairwise correlation:


,META/NVDA,GS/WFC,GOOGL/NVDA,MDT/PFE,PEP/YUM
META/NVDA,1.00,-0.05,0.21,0.02,-0.09
GS/WFC,-0.05,1.00,0.01,-0.09,-0.02
GOOGL/NVDA,0.21,0.01,1.00,-0.02,-0.09
MDT/PFE,0.02,-0.09,-0.02,1.00,0.02
PEP/YUM,-0.09,-0.02,-0.09,0.02,1.00
